In [0]:
q = """
SELECT * FROM 4_prod.dicom_tags.extracted_dicom_tags 
WHERE name = 'SOPClassUID' 
AND value IN ('1.2.840.10008.5.1.4.1.1.4', '1.2.840.10008.5.1.4.1.1.2')
AND dicom_path ILIKE '%Hackathon3%'
"""
df_sop = spark.sql(q)
display(df_sop.limit(100))

In [0]:
df = spark.read.parquet("/Volumes/1_inland/sectra/vone/Hackathon3_output_v3_batch0.parquet/")
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

schema = StructType([
    StructField("is_text_detected", StringType(), True),
    StructField("is_personal_information_detected", StringType(), True),
    StructField("confidence_score", IntegerType(), True)
])

df_parsed = (
    df
    .withColumn("parsed", F.from_json(F.col("llm_result"), schema))
    .select(
        "*",
        F.col("parsed.is_text_detected").alias("is_text_detected"),
        F.col("parsed.is_personal_information_detected").alias("is_personal_information_detected"),
        F.col("parsed.confidence_score").alias("confidence_score")
    )
    .drop("parsed")
)


display(df_parsed)


In [0]:
df_joined = df_sop.join(df_parsed, df_sop.dicom_path == df_parsed.path, "inner")
display(df_joined)